# Inference Optimization & Model Serving - Start Here

Welcome to Phase 30 of the Zero-to-AI curriculum. This module covers how to make LLM inference fast, cheap, and scalable: the techniques that turn a trained model into a production-grade service.

**Duration:** 6-8 hours right now, with planned expansion

> Status: this phase is intentionally published as a work in progress.
> Today, the reliable starting point is `03_serving_with_vllm.ipynb`. The remaining planned notebooks show the intended roadmap, not completed coverage.

---

## Prerequisites

- Completion of `14-local-llms/`
- Completion of `04-token/`
- Basic understanding of PyTorch devices and CUDA memory

## How To Use This Phase Right Now

1. Treat this as an introduction to serving and optimization, not a complete mastery path.
2. Start with the available vLLM notebook and use it to learn batching, runtime behavior, and measurement.
3. Pair this phase with `09-mlops/` and `14-local-llms/` for a more complete serving picture.

## Why Inference Optimization Matters

A 7B parameter model in FP16 uses ~14 GB of VRAM just to load. Every token it generates requires reading those weights plus the growing KV cache. At scale, inference cost dominates - often 10–100× the cost of training.

This phase teaches the techniques that reduce latency and cost:

```
Technique                    Speedup      Memory Savings
──────────────────────────────────────────────────────────
KV Cache + PagedAttention    2–4×         30–50% less waste
INT4 Quantization (AWQ)      2–3×         ~75% model size reduction
Continuous Batching          5–10× throughput   -
Speculative Decoding         1.5–2.5×     minimal overhead
Prefix Caching               1.3–2×       reuse repeated prompts
```

## Learning Path

| # | Notebook | Topic | Status |
|---|---------|-------|--------|
| 01 | `01_kv_cache_paged_attention.ipynb` | Visualizing and managing the KV cache | Planned |
| 02 | `02_quantization_deep_dive.ipynb` | Quantizing models from FP16 to INT4 (AWQ) | Planned |
| 03 | `03_serving_with_vllm.ipynb` | vLLM-based serving and batching | ✅ Available |
| 04 | `04_speculative_decoding.ipynb` | Speeding up inference with draft models | Planned |

### Start with the available notebook:

**`03_serving_with_vllm.ipynb`** - Set up a vLLM server, understand continuous batching, and measure throughput.

### Key Concepts You'll Learn

- **KV Cache**: Why autoregressive generation is memory-bound, not compute-bound
- **PagedAttention**: How vLLM avoids memory fragmentation (like virtual memory for attention)
- **Quantization**: AWQ, GPTQ, EXL2, GGUF - tradeoffs between quality and speed
- **Continuous Batching**: Serving many requests simultaneously without waiting for the longest
- **Speculative Decoding**: Using a small fast model to draft tokens, verified by the large model
- **Serving Runtimes**: vLLM vs TensorRT-LLM vs SGLang vs TGI

## Key Metrics

When benchmarking inference, measure these:

| Metric | What It Measures | Target |
|--------|-----------------|--------|
| **TTFT** (Time to First Token) | Latency until first output token | < 500ms for chat |
| **TPS** (Tokens per Second) | Decode speed | 30-100+ tokens/sec |
| **Throughput** | Total tokens/sec across all concurrent requests | Maximize |
| **Memory** | Peak VRAM usage | Fit your GPU budget |
| **Cost per 1M tokens** | Cost per million tokens served | Minimize |

---

## API Provider Speed Benchmarks (April 2026)

If you're serving via API providers rather than self-hosting, the provider you choose matters enormously. [Artificial Analysis](https://artificialanalysis.ai/leaderboards/providers) benchmarks 23+ providers on the same models.

### gpt-oss-120B Speed Across Providers

| Provider | Output Speed (tok/s) | Relative to Slowest |
|----------|---------------------|---------------------|
| **Cerebras** | 1,833 | 9.2x |
| **SambaNova** | 680 | 3.4x |
| **Nebius (Fast)** | 653 | 3.3x |
| **Groq** | 421 | 2.1x |
| **Together** | 364 | 1.8x |
| **Fireworks** | 329 | 1.6x |
| **Nebius** | 283 | 1.4x |
| **Lepton** | 223 | 1.1x |
| Baseline | ~200 | 1.0x |

> **Key insight**: The same model (gpt-oss-120B) can be **9x faster** on Cerebras vs baseline providers. This is more impactful than most algorithmic optimizations!

### Speed vs Intelligence Across Models

| Model | Intelligence Index | Speed (tok/s) | Price ($/1M) |
|-------|-------------------|---------------|--------------|
| gpt-oss-20B | 24 | 273 | $0.10 |
| gpt-oss-120B | 33 | 209 | $0.26 |
| Gemini 3 Flash | 46 | 158 | $1.13 |
| Nemotron 3 Super | 36 | 155 | $0.41 |
| GPT-5.4 mini (xhigh) | 49 | 146 | $1.69 |
| Gemini 3.1 Pro Preview | 57 | 119 | $4.50 |
| Claude 4.5 Haiku | 37 | 98 | $2.00 |
| GPT-5.4 (xhigh) | 57 | 80 | $5.63 |
| GPT-5.5 (xhigh) | 60 | 72 | $11.25 |
| Claude Opus 4.7 (max) | 57 | 48 | $10.00 |

*Data from [artificialanalysis.ai](https://artificialanalysis.ai) - April 2026*

### Optimization Decision Tree

```
Is it worth self-hosting?
├─ < 1M tokens/day → Use API (cheaper, simpler)
│   ├─ Latency-critical → Cerebras / Groq (fastest providers)
│   ├─ Budget-first → DeepSeek V4 Flash via API ($0.17/1M)
│   └─ Quality-first → GPT-5.4 or Claude Sonnet 4.6 via API
│
└─ > 1M tokens/day → Consider self-hosting
    ├─ Single GPU (24GB) → Quantized gpt-oss-20B or Gemma 4 E4B
    ├─ Multi GPU (80GB+) → gpt-oss-120B with vLLM
    └─ GPU cluster → DeepSeek V4 Flash (685B MoE, ~45GB active)
```